In [1]:
!pip install opendatasets --quiet
import opendatasets as od
od.download('https://www.kaggle.com/datasets/saurabhshahane/twitter-sentiment-dataset')

Dataset URL: https://www.kaggle.com/datasets/saurabhshahane/twitter-sentiment-dataset


100%|██████████| 7.60M/7.60M [00:00<00:00, 67.4MB/s]

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/content/twitter-sentiment-dataset/Twitter_Data.csv')
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [ ]:
df.iloc[100]['clean_text']

'why limited here are other prefixes for twitter that perhaps more accurately capture the state the citizens '

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB


In [9]:
df.dropna(inplace=True)
len(df)

162969

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

In [ ]:
# test vectorizer
vectorizer = CountVectorizer(max_features=10000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['category']
X.shape, y.shape

In [ ]:
!pip install mlflow

In [ ]:
import mlflow
import mlflow.sklearn

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


# run ngrok http 5000 in local
TRACKING_URI = "https://b31880069539.ngrok-free.app"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("sentiment analysis")

def run_experiment(
  df,
  vectorizer_type="tfidf",
  ngram_range=(1, 1),
  vectorizer_max_features=5000,

  model_type="logistic",
  model_params=None,

  test_size=0.2,
):
  """
  Run 1 experiment sentiment analysis với hyperparameter tuning support
  """

  default_params = {
    "logistic": {
      "C": 1.0,              # regularization strength
      "max_iter": 1000,
      "solver": "lbfgs"
    },
    "nb": {
      "alpha": 1.0           # smoothing
    },
    "svm": {
      "C": 1.0               # margin control
    }
  }


  if model_params is None:
    model_params = default_params[model_type]
  else:
    model_params = {**default_params[model_type], **model_params}

  X_train_text, X_test_text, y_train, y_test = train_test_split(df['clean_text'], df['category'], test_size=test_size, random_state=42)

  if vectorizer_type == "bow":
    vectorizer = CountVectorizer(
      ngram_range=ngram_range,
      max_features=vectorizer_max_features
    )
  elif vectorizer_type == "tfidf":
    vectorizer = TfidfVectorizer(
      ngram_range=ngram_range,
      max_features=vectorizer_max_features
    )
  else:
    raise ValueError("invalid vectorizer_type")

  X_train = vectorizer.fit_transform(X_train_text)
  X_test = vectorizer.transform(X_test_text)

  if model_type == "logistic":
    model = LogisticRegression(**model_params)
  elif model_type == "nb":
    model = MultinomialNB(**model_params)
  elif model_type == "svm":
    model = LinearSVC(**model_params)
  else:
    raise ValueError("invalid model_type")

  with mlflow.start_run():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    mlflow.log_param("vectorizer_type", vectorizer_type)
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", vectorizer_max_features)

    mlflow.log_param("model_type", model_type)
    for k, v in model_params.items():
      mlflow.log_param(k, v)

    # metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    # model
    mlflow.sklearn.log_model(model, "model")

  return {
    "accuracy": acc,
    "f1_score": f1,
    "model": model,
    "vectorizer": vectorizer
  }

# pipeline

In [ ]:
vectorizer = CountVectorizer(max_features=10000)
model = LogisticRegression(max_iter=1000)
pipeline = Pipeline([
            ("vectorizer", vectorizer),
            ("model", model)
        ])

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'],
    df['category'],
    test_size=0.2,
    random_state=42
  )

In [ ]:
pipeline.fit(X_train_text, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer(max_features=10000)),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
pred = pipeline.predict(X_test_text)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='weighted')
acc, f1

(0.9455114438240166, 0.9453119026293918)

In [ ]:
!pip install onnx onnxruntime skl2onnx

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

initial_types = [("input", StringTensorType([None, 1]))]   # Input shape = (batch_size, 1)

# Convert to ONNX
onnx_model = convert_sklearn(
  pipeline,
  initial_types=initial_types
)

In [ ]:
onnx_path = "onnx_pipeline.onnx"
with open(onnx_path, 'wb') as f:
  f.write(onnx_model.SerializeToString())

In [ ]:
import onnxruntime as ort
import numpy as np

texts = ['What is that', 'I hate you', 'I love you']
session = ort.InferenceSession(onnx_path)
inputs = np.array(texts).reshape(-1, 1)   # Input shape = (batch_size, 1)
outputs = session.run(None, {"input": inputs})
outputs

[array([ 0, -1,  1], dtype=int64),
 [{-1: 0.022375352680683136, 0: 0.9522366523742676, 1: 0.025387974455952644},
  {-1: 0.9369741678237915, 0: 0.0628664642572403, 1: 0.00015939986042212695},
  {-1: 0.002951366826891899, 0: 0.054115090519189835, 1: 0.9429335594177246}]]

In [ ]:
def _to_sentiment(label: int):
        if label == 1:
            return 'positive'
        elif label ==-1:
            return 'negative'

        return 'neutral'
formated_outputs =  [
                    {
                        'text': text,
                        'sentiment': _to_sentiment(sentiment),
                        'probabilities': prob,
                    }
                    for text, sentiment, prob in zip(texts, outputs[0], outputs[1])
                ]
formated_outputs

[{'text': 'What is that',
  'sentiment': 'neutral',
  'probabilities': {-1: 0.022375352680683136,
   0: 0.9522366523742676,
   1: 0.025387974455952644}},
 {'text': 'I hate you',
  'sentiment': 'negative',
  'probabilities': {-1: 0.9369741678237915,
   0: 0.0628664642572403,
   1: 0.00015939986042212695}},
 {'text': 'I love you',
  'sentiment': 'positive',
  'probabilities': {-1: 0.002951366826891899,
   0: 0.054115090519189835,
   1: 0.9429335594177246}}]

# Get best model

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

experiment = mlflow.get_experiment_by_name("sentiment analysis")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"]  # sort by f1
)

# lấy run tốt nhất
best_run = runs.iloc[0]

run_id = best_run.run_id
print("Best run:", run_id)

# register model
model_uri = f"runs:/{run_id}/model"

model_name = "sentiment_model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Registered:", registered_model.name)

In [5]:
LABEL_MAP = {
    -1: "negative",
    0: "neutral",
    1: "positive",
}

In [18]:
reference_distribution = (
        df["category"]
        .map(lambda value: LABEL_MAP.get(value, str(value)))
        .value_counts(normalize=True)
        .sort_index()
        .apply(float)
        .to_dict()
    )
reference_distribution

{'negative': 0.2178880646012432,
 'neutral': 0.33878222238585254,
 'positive': 0.4433297130129043}

In [15]:
records = [{
                "text": "fddddddddddddddd",
                "predicted_label": "positive",
           },
           {
               "text": "hahaha, dd",
               "predicted_label": "negative"
           },
           {
               "text": "1",
               "predicted_label": "neutral"
           },
           {
               "text": "2",
               "predicted_label": "positive"
           },
           {
               "text": "3",
               "predicted_label": "negative"
           },
           {
               "text": "4",
               "predicted_label": "positive"
           },
           {
               "text": "5",
               "predicted_label": "negative"
           },
]

In [12]:
from collections import Counter
def _normalize_distribution(values):
    counter = Counter(values)
    total = sum(counter.values()) or 1
    return {label: count / total for label, count in counter.items()}

In [16]:
predicted_distribution = _normalize_distribution(record["predicted_label"] for record in records)
predicted_distribution

{'positive': 0.42857142857142855,
 'negative': 0.42857142857142855,
 'neutral': 0.14285714285714285}

In [19]:
all_labels = sorted(set(reference_distribution) | set(predicted_distribution))
all_labels

['negative', 'neutral', 'positive']

In [22]:
distribution_shift = 0.0
for label in all_labels:
  distribution_shift += abs(predicted_distribution.get(label, 0.0) - reference_distribution.get(label, 0.0))